# G1 stand-still training on Colab

Trains `Unitree-G1-Stand` (`g1_app/training/stand/`) on a Colab GPU.
Two Colab adaptations baked in:

- **Checkpoints live on Google Drive** (Colab wipes `/content` on disconnect) via a symlink.
- **No dashboard** (`--no-dashboard`); monitor with Colab's TensorBoard cell instead.

Workflow: run cells top to bottom. Train in **chunks** (`--max-iterations 600`, then `--resume` with a higher target) — Colab kills sessions, checkpoints every 100 iters make that survivable.

Prereqs: Runtime → Change runtime type → **GPU** (T4 is fine). Push this repo to GitHub first (it's already git-initialized with submodules + LFS); Colab clones it fresh each session, so reward/config edits flow via `git push` → `git pull` — no zip juggling.

In [ ]:
# Config — edit these to taste
REPO_URL = "https://github.com/tal2k/UniRobot.git"  # your GitHub repo
BRANCH = "master"
WORKDIR = "/content/UniRobot"                              # where it clones
DRIVE_LOGS = "/content/drive/MyDrive/UniRobot/logs"        # Drive-backed logs dir
NUM_ENVS = 512        # T4 (16 GB): 512. A100/Pro: 1024. OOM -> 256.
MAX_ITERS = 600       # one chunk; raise on each resume (600 -> 1200 -> 2000 ...)
#
# Private repo? Uncomment, enter a fine-grained PAT (read-only is enough),
# never hardcode the token in the notebook:
# from getpass import getpass
# REPO_URL = f"https://{getpass('GitHub token:')}@github.com/tal2k/UniRobot.git"

In [ ]:
# Cell 1 — GPU check + Drive mount (rerun after every disconnect)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — clone (first time) or pull (later sessions). /content is wiped
# on disconnect, so this cell restores exact code.
# Only unitree_rl_mjlab is initialized: training needs nothing else
# (g1_app uses pip mujoco + mjlab; the other vendored trees sit unused).
import os
%cd /content
if not os.path.isdir(WORKDIR + "/.git"):
    import shutil; shutil.rmtree(WORKDIR, ignore_errors=True)  # clear failed clones
    !GIT_LFS_SKIP_SMUDGE=1 git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"
    %cd "$WORKDIR"
else:
    %cd "$WORKDIR"
    !git pull --rebase
# --force re-checks-out files even when the recorded SHA looks right.
!git submodule update --init --force unitree_rl_mjlab
!git rev-parse --short HEAD && git -C unitree_rl_mjlab log --oneline -1
!ls unitree_rl_mjlab/scripts/train.py g1_app/cli.py g1_app/training/stand/stand_env_cfg.py


In [ ]:
# Cell 3 — install (rerun after every disconnect)
%cd "$WORKDIR"
!pip -q install -e "./g1_app[train]"
# One-line compat patch (proven-local build): PyPI mjlab==1.2.0 calls
# wp.context.runtime (absent in warp-lang 1.14/1.17); local venv uses
# wp.get_cuda_driver_version() instead. Whole mjlab tree is otherwise identical.
!SIM=$(pip show -f mjlab 2>/dev/null | grep Location | awk '{print $2}')/mjlab/sim/sim.py && sed -i 's/wp\.context\.runtime\.driver_version/wp.get_cuda_driver_version()/' "$SIM" && grep -n "driver_ver =" "$SIM"
import torch
print("cuda:", torch.cuda.is_available())
import mujoco as _mj
print("mujoco:", _mj.__version__, "(must be 3.5.0)")
# If cuda is False, pip overwrote torch — run this, then rerun this cell:
# !pip -q install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Cell 4 — Drive-backed logs (MUST run before any training cell).
# lab/train.py always writes to unitree_rl_mjlab/logs/, so symlink it to Drive.
!mkdir -p "$DRIVE_LOGS"
!ln -sfn "$DRIVE_LOGS" "$WORKDIR/unitree_rl_mjlab/logs"
!ls -la "$WORKDIR/unitree_rl_mjlab/" | grep logs

## Smoke test (minutes)

Validates the whole pipeline before committing hours. Expect iters ticking and a run folder under `unitree_rl_mjlab/logs/rsl_rl/g1_stand/`. If this fails, fix it first — do not start a long run.

In [ ]:
# Cell 5 — smoke test
%cd "$WORKDIR"
!python3 -m g1_app.cli train-stand -- --num-envs 128 --max-iterations 30 --no-dashboard

## Training chunks (hours each)

Keep the tab open. When the chunk finishes — or Colab kills the session — rerun cells 1–4, then the resume cell below with a higher `MAX_ITERS`. Never restart from scratch: `--resume` continues from the latest checkpoint.

`PUSH_PHASE` note: the repo default is Phase 1 (gentle shoves). Flip `PUSH_PHASE` to 2 in `g1_app/training/stand/stand_env_cfg.py` once videos show quiet standing — commit + push locally, then rerun the clone cell (`git pull`) here. Never edit under `/content` directly: it dies with the session.

In [ ]:
# Cell 6 — training chunk (fresh or continued via MAX_ITERS; add --resume to continue)
%cd "$WORKDIR"
!python3 -m g1_app.cli train-stand -- --num-envs $NUM_ENVS --max-iterations $MAX_ITERS --no-dashboard

In [ ]:
# Cell 7 — resume from latest checkpoint (after rerunning cells 1–4).
# Bump MAX_ITERS in the config cell first (e.g. 600 -> 1200).
%cd "$WORKDIR"
!python3 -m g1_app.cli train-stand -- --num-envs $NUM_ENVS --max-iterations $MAX_ITERS --no-dashboard --resume

## Monitor

Run in its own cell while training. Watch **Episode_Reward/stand_success → 3.0**. Stage-II terms (`leg_pose`, `feet_width`, `ang_vel_damp`) appear as their own tags automatically.

In [ ]:
# Cell 8 — TensorBoard (points at Drive, so it works across sessions)
%load_ext tensorboard
%tensorboard --logdir "$DRIVE_LOGS"

## Getting the policy home

Checkpoints + `policy.onnx` land every 100 iters under `logs/rsl_rl/g1_stand/<run>/` — already on your Drive via the symlink. Download `policy.onnx`, place it with your local runs, and use it via `g1 stand --stand-policy <path>` or `g1 record -- --experiment g1_stand --stand`.

## Troubleshooting

- **CUDA out of memory** → `NUM_ENVS = 256`.
- **Session dies constantly** → smaller `MAX_ITERS` (300), or Colab Pro.
- **MuJoCo GL errors** → training doesn't render; if they appear, prefix commands with `MUJOCO_GL=egl`.
- **Lost edits** (`PUSH_PHASE`, reward weights) → you edited `/content` instead of the repo. Edit locally, push, rerun the clone cell.